# 01 - Dataset Overview and City Selection

**Purpose:** choose a city-level scope for the Yelp community-attention forecasting project.

The business file is used first because it contains city/state fields and is much smaller than the review and user files.

## Workflow

1. Verify the expected raw Yelp files.
2. Count businesses by normalized city/state.
3. Save the city-count table.
4. Select the working city for the rest of the pipeline.

In [1]:
from pathlib import Path
import csv
import json
from collections import Counter
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_YELP_DIR = DATA_DIR / "raw" / "yelp"
INTERIM_DIR = DATA_DIR / "interim"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

BUSINESS_PATH = RAW_YELP_DIR / "yelp_academic_dataset_business.json"
REVIEW_PATH = RAW_YELP_DIR / "yelp_academic_dataset_review.json"
USER_PATH = RAW_YELP_DIR / "yelp_academic_dataset_user.json"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)

C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp


In [2]:
expected_files = [
    "yelp_academic_dataset_business.json",
    "yelp_academic_dataset_review.json",
    "yelp_academic_dataset_user.json",
    "yelp_academic_dataset_checkin.json",
    "yelp_academic_dataset_tip.json",
]

for filename in expected_files:
    path = RAW_YELP_DIR / filename
    size_mb = path.stat().st_size / (1024 * 1024) if path.exists() else 0
    print(f"{filename:<40} exists={path.exists():<5} size_mb={size_mb:,.1f}")

yelp_academic_dataset_business.json      exists=1     size_mb=113.4
yelp_academic_dataset_review.json        exists=1     size_mb=5,094.4
yelp_academic_dataset_user.json          exists=1     size_mb=3,207.5
yelp_academic_dataset_checkin.json       exists=1     size_mb=273.7
yelp_academic_dataset_tip.json           exists=1     size_mb=172.2


In [3]:
def iter_jsonl(path):
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            try:
                yield json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {path} at line {line_number}") from exc

city_state_counts = Counter()

for record in iter_jsonl(BUSINESS_PATH):
    city = (record.get("city") or "").strip()
    state = (record.get("state") or "").strip()
    if city:
        city_state_counts[(city, state)] += 1

city_rows = [
    {"city": city, "state": state, "business_count": count}
    for (city, state), count in city_state_counts.items()
]
city_rows = sorted(city_rows, key=lambda row: (-row["business_count"], row["city"], row["state"]))

city_counts_path = OUTPUTS_DIR / "city_business_counts.csv"
with city_counts_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["city", "state", "business_count"])
    writer.writeheader()
    writer.writerows(city_rows)

print(f"Saved: {city_counts_path}")
print("\nTop 20 city/state combinations:")
for row in city_rows[:20]:
    print(f"{row['city']:<24} {row['state']:<4} {row['business_count']:>8}")

Saved: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\outputs\city_business_counts.csv

Top 20 city/state combinations:
Philadelphia             PA      14568
Tucson                   AZ       9251
Tampa                    FL       9049
Indianapolis             IN       7543
Nashville                TN       6971
New Orleans              LA       6208
Reno                     NV       5934
Edmonton                 AB       5054
Saint Louis              MO       4828
Santa Barbara            CA       3834
Boise                    ID       2938
Clearwater               FL       2221
Saint Petersburg         FL       1663
Metairie                 LA       1644
Sparks                   NV       1624
Wilmington               DE       1447
Franklin                 TN       1411
St. Louis                MO       1254
St. Petersburg           FL       1185
Meridian                 ID       1043


## Decision

The project uses **New Orleans, Louisiana**. It has enough businesses for time-series, SNA, and NLP analysis while remaining manageable for an academic notebook workflow.

## Note

The exploratory count found **6,208** New Orleans businesses. The later extraction uses stricter normalization and selects **6,215** businesses. The normalized count is used throughout the project.